In [2]:
import pandas as pd
import numpy as np

# --- 1. SETUP & DATA LOADING ---
# Using the exact filename from your VS Code sidebar
file_path = 'Most-Popular-Programming-Languages.csv'
df = pd.read_csv(file_path)

# Standardize column names by removing leading/trailing spaces
df.columns = df.columns.str.strip()

# Set Month as index
# Note: If your CSV uses 'Date', change 'Month' to 'Date' below
df['Month'] = pd.to_datetime(df['Month'])
df.set_index('Month', inplace=True)

# --- 2. DYNAMIC LANGUAGE ASSIGNMENT ---
# This finds the actual column names (e.g., "React Worldwide (%)") 
# based on your assigned index 7 (React) and 8 (Swift)
try:
    lang_a = [col for col in df.columns if 'React' in col][0]
    lang_b = [col for col in df.columns if 'Swift' in col][0]
    print(f"Targeting Columns: {lang_a} and {lang_b}")
except IndexError:
    print("Error: Could not find columns containing 'React' or 'Swift'.")

# --- 3. TASK 1: GROWTH ANALYSIS (For React) ---
# Compute month-to-month growth rate
df['Growth_Rate'] = df[lang_a].pct_change() * 100

# Rolling statistics (6-month window as per Lab instructions)
df['Moving_Avg'] = df[lang_a].rolling(window=6).mean()
df['Moving_STD'] = df[lang_a].rolling(window=6).std()

# Classify growth phases using NumPy
conditions = [(df['Growth_Rate'] > 5), (df['Growth_Rate'] < -5)]
choices = ['Growth', 'Decline']
df['Phase'] = np.select(conditions, choices, default='Stable')

# --- 4. TASK 2: LIFECYCLE CLASSIFICATION ---
mean_growth = df['Growth_Rate'].mean()
std_growth = df['Growth_Rate'].std()

lifecycle_conds = [
    (df['Growth_Rate'] > 0) & (df['Growth_Rate'] < mean_growth), # Introduction
    (df['Growth_Rate'] > mean_growth),                           # Growth
    (df['Growth_Rate'].abs() <= 1),                             # Maturity
    (df['Growth_Rate'] < 0) & (df['Growth_Rate'] < (-1 * std_growth)) # Decline
]
lifecycle_choices = ['Introduction', 'Growth', 'Maturity', 'Decline']
df['Lifecycle_Phase'] = np.select(lifecycle_conds, lifecycle_choices, default='Maturity')

# --- 5. TASK 3: COMPARATIVE METRICS ---
correlation = df[lang_a].corr(df[lang_b])
dominance_ratio = (df[lang_a] > df[lang_b]).mean() * 100
rpi = df[lang_a].mean() / df[lang_b].mean()

# --- 6. OUTPUTS ---
print("\n" + "="*30)
print(f"STATISTICAL SUMMARY: {lang_a}")
print("="*30)
print(df[lang_a].describe())

print("\n--- Phase Counts ---")
print(df['Phase'].value_counts())

print("\n--- Comparative Metrics (React vs Swift) ---")
print(f"Correlation Coefficient: {correlation:.4f}")
print(f"Dominance Ratio: {dominance_ratio:.2f}%")
print(f"Relative Performance Index: {rpi:.4f}")

# Preview the results table
print("\n--- Final Classification Preview ---")
cols_to_show = [lang_a, 'Growth_Rate', 'Phase', 'Lifecycle_Phase']
print(df[cols_to_show].tail(10))

Targeting Columns: React Worldwide(%) and Swift Worldwide(%)

STATISTICAL SUMMARY: React Worldwide(%)
count    249.000000
mean      25.883534
std       32.518210
min        1.000000
25%        1.000000
50%        2.000000
75%       50.000000
max      100.000000
Name: React Worldwide(%), dtype: float64

--- Phase Counts ---
Phase
Stable     171
Growth      53
Decline     25
Name: count, dtype: int64

--- Comparative Metrics (React vs Swift) ---
Correlation Coefficient: 0.4503
Dominance Ratio: 35.74%
Relative Performance Index: 0.8437

--- Final Classification Preview ---
            React Worldwide(%)  Growth_Rate    Phase Lifecycle_Phase
Month                                                               
2023-12-01                  73   -12.048193  Decline        Maturity
2024-01-01                  89    21.917808   Growth          Growth
2024-02-01                  98    10.112360   Growth          Growth
2024-03-01                  91    -7.142857  Decline        Maturity
2024-04-0